# 电信客户流失预测与价值细分系统

**课程：** 数据分析与数据挖掘  
**题目：** 电信客户流失预测与价值细分系统  
**学生：** 林乐珊  
**学号：** 202425310809  

## Notebook 说明

本 Notebook 是课程项目的完整可复现实验代码，覆盖：

1. 数据读取与数据清洗；
2. 探索性数据分析；
3. 特征工程；
4. Logistic Regression + SMOTE、Random Forest、Gradient Boosting 模型训练；
5. ROC-AUC、PR-AUC、Precision、Recall、F1 等指标评估；
6. ROC 曲线、PR 曲线、混淆矩阵可视化；
7. Gradient Boosting 超参数敏感性分析；
8. SHAP 全局解释分析；
9. K-Means 客户价值细分；
10. 单客户流失风险推理函数；
11. 图表、指标表和模型文件保存。

> 数据集使用公开的 Telco Customer Churn 数据集。若本地没有数据文件，请将 `WA_Fn-UseC_-Telco-Customer-Churn.csv` 或 `Telco-Customer-Churn.csv` 放到 `data/raw/` 目录下；代码也会尝试从公开镜像地址自动下载。

## 1. 环境与依赖

如本地环境缺少依赖，可先运行下面命令安装：

```bash
pip install -r requirements.txt
```

本 Notebook 依赖的核心包包括：

- numpy
- pandas
- scikit-learn
- imbalanced-learn
- matplotlib
- seaborn
- shap
- joblib

In [ ]:
# 如需在新环境中安装依赖，可取消下面注释后运行。
# !pip install numpy pandas scikit-learn imbalanced-learn matplotlib seaborn shap joblib -q

## 2. 导入依赖与全局配置

In [ ]:
from __future__ import annotations

import os
import random
import urllib.request
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font="SimHei", rc={"axes.unicode_minus": False})

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR, MODEL_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2


def set_seed(seed: int = RANDOM_STATE) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(RANDOM_STATE)
print(f"Project directory: {PROJECT_DIR}")
print(f"Data directory: {DATA_DIR}")

## 3. 数据准备与读取

本项目使用 Telco Customer Churn 数据集。代码将依次尝试：

1. 从 `data/raw/` 中读取本地数据；
2. 从当前目录读取数据；
3. 从常见公开镜像地址下载数据。

如果自动下载失败，请手动下载数据并放到：

```text
data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
```

In [ ]:
DATA_FILENAMES = [
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "Telco-Customer-Churn.csv",
    "telco_churn.csv",
]

PUBLIC_DATA_URLS = [
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/telco-churn.csv",
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/4-Classification/data/WA_Fn-UseC_-Telco-Customer-Churn.csv",
]

REQUIRED_COLUMNS = {
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
}


def validate_telco_dataframe(df: pd.DataFrame) -> bool:
    """Return True if dataframe columns match the Telco churn schema."""
    return REQUIRED_COLUMNS.issubset(set(df.columns))


def find_local_data_file() -> Optional[Path]:
    """Find Telco churn CSV from common local paths."""
    candidate_paths = []
    for filename in DATA_FILENAMES:
        candidate_paths.extend([
            DATA_DIR / filename,
            PROJECT_DIR / filename,
            Path(filename),
        ])

    for path in candidate_paths:
        if path.exists():
            try:
                df_head = pd.read_csv(path, nrows=5)
                if validate_telco_dataframe(df_head):
                    return path
            except Exception:
                continue
    return None


def download_telco_data() -> Path:
    """Download Telco churn data from public mirrors."""
    output_path = DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
    last_error = None

    for url in PUBLIC_DATA_URLS:
        try:
            print(f"Trying to download data from: {url}")
            urllib.request.urlretrieve(url, output_path)
            df_head = pd.read_csv(output_path, nrows=5)
            if validate_telco_dataframe(df_head):
                print(f"Downloaded valid dataset to: {output_path}")
                return output_path
        except Exception as exc:
            last_error = exc
            print(f"Download failed: {exc}")

    raise FileNotFoundError(
        "未找到 Telco Customer Churn 数据集，且自动下载失败。"
        "请手动下载 WA_Fn-UseC_-Telco-Customer-Churn.csv 并放入 data/raw/ 目录。"
        f"最后一次错误：{last_error}"
    )


def load_raw_data() -> pd.DataFrame:
    """Load raw Telco churn data."""
    data_path = find_local_data_file()
    if data_path is None:
        data_path = download_telco_data()

    df_raw = pd.read_csv(data_path)
    if not validate_telco_dataframe(df_raw):
        missing = REQUIRED_COLUMNS - set(df_raw.columns)
        raise ValueError(f"数据字段不完整，缺少字段：{missing}")

    print(f"Loaded data from: {data_path}")
    print(f"Raw shape: {df_raw.shape}")
    return df_raw


raw_df = load_raw_data()
raw_df.head()

## 4. 数据清洗与特征工程

关键处理包括：

- `TotalCharges` 从字符串转换为数值；
- 将空白 `TotalCharges` 填充为 0；
- 将 `Churn` 转换为二值标签；
- 构造平均月消费、单位在网费用、在网时长分段等衍生特征；
- 删除 `customerID`，避免无意义 ID 进入模型。

In [ ]:
def clean_and_engineer_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Clean raw Telco data and build derived features."""
    data = df_raw.copy()

    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    missing_total_charges = data["TotalCharges"].isna().sum()
    data["TotalCharges"] = data["TotalCharges"].fillna(0.0)

    data["ChurnLabel"] = data["Churn"].map({"Yes": 1, "No": 0}).astype(int)

    data["AvgMonthlyCharge"] = np.where(
        data["tenure"] > 0,
        data["TotalCharges"] / data["tenure"],
        0.0,
    )
    data["ChargePerTenure"] = data["MonthlyCharges"] / (data["tenure"] + 1)

    data["TenureSegment"] = pd.cut(
        data["tenure"],
        bins=[-1, 6, 12, 24, 48, 72],
        labels=["0-6", "7-12", "13-24", "25-48", "49-72"],
    ).astype(str)

    print(f"TotalCharges blank/missing fixed: {missing_total_charges}")
    return data


df = clean_and_engineer_features(raw_df)

basic_stats = pd.DataFrame({
    "项目": [
        "样本数量",
        "原始字段数",
        "流失客户数",
        "未流失客户数",
        "流失率",
        "TotalCharges 缺失/空白修正",
    ],
    "数值": [
        len(df),
        raw_df.shape[1],
        int(df["ChurnLabel"].sum()),
        int((1 - df["ChurnLabel"]).sum()),
        f"{df['ChurnLabel'].mean():.2%}",
        int((pd.to_numeric(raw_df["TotalCharges"], errors="coerce").isna()).sum()),
    ],
})

basic_stats.to_csv(TABLE_DIR / "basic_stats.csv", index=False, encoding="utf-8-sig")
basic_stats

## 5. 探索性数据分析

In [ ]:
def save_current_figure(filename: str) -> None:
    """Save current matplotlib figure to the figure directory."""
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved figure: {path}")


plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="Churn", order=["No", "Yes"])
ax.set_title("客户流失类别分布")
ax.set_xlabel("是否流失")
ax.set_ylabel("客户数量")
save_current_figure("01_churn_distribution.png")
plt.show()

plt.figure(figsize=(7, 4))
ax = sns.histplot(data=df, x="tenure", hue="Churn", bins=30, kde=True, multiple="stack")
ax.set_title("在网时长与流失分布")
ax.set_xlabel("在网时长（月）")
ax.set_ylabel("客户数量")
save_current_figure("02_tenure_distribution.png")
plt.show()

plt.figure(figsize=(7, 4))
ax = sns.boxplot(data=df, x="Churn", y="MonthlyCharges", order=["No", "Yes"])
ax.set_title("月费用与流失关系")
ax.set_xlabel("是否流失")
ax.set_ylabel("月费用")
save_current_figure("03_monthly_charges_boxplot.png")
plt.show()

contract_churn = (
    df.groupby("Contract")["ChurnLabel"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=contract_churn, x="Contract", y="ChurnLabel")
ax.set_title("不同合约类型的流失率")
ax.set_xlabel("合约类型")
ax.set_ylabel("流失率")
save_current_figure("04_contract_churn_rate.png")
plt.show()

contract_churn

## 6. 建模数据准备

使用分层划分保证训练集和测试集的流失比例一致。  
预处理管线统一处理数值变量和类别变量，避免数据泄漏。

In [ ]:
TARGET_COLUMN = "ChurnLabel"
DROP_COLUMNS = ["customerID", "Churn", TARGET_COLUMN]

X = df.drop(columns=DROP_COLUMNS, errors="ignore")
y = df[TARGET_COLUMN].astype(int)

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}, Test churn rate: {y_test.mean():.2%}")


def build_onehot_encoder() -> OneHotEncoder:
    """Build OneHotEncoder compatible with multiple scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor() -> ColumnTransformer:
    """Build preprocessing pipeline for numeric and categorical columns."""
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", build_onehot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )

## 7. 模型训练与评估函数

In [ ]:
def build_models() -> Dict[str, BaseEstimator]:
    """Build candidate models with unified preprocessing."""
    models = {
        "LogisticRegression_SMOTE": ImbPipeline(steps=[
            ("preprocess", build_preprocessor()),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("model", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "RandomForest": Pipeline(steps=[
            ("preprocess", build_preprocessor()),
            ("model", RandomForestClassifier(
                n_estimators=120,
                max_depth=10,
                min_samples_leaf=8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "GradientBoosting": Pipeline(steps=[
            ("preprocess", build_preprocessor()),
            ("model", GradientBoostingClassifier(
                n_estimators=120,
                learning_rate=0.05,
                max_depth=3,
                random_state=RANDOM_STATE,
            )),
        ]),
    }
    return models


def predict_positive_probability(model: BaseEstimator, features: pd.DataFrame) -> np.ndarray:
    """Return predicted probability of positive class."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]

    decision_score = model.decision_function(features)
    return 1 / (1 + np.exp(-decision_score))


def evaluate_binary_classifier(
    model: BaseEstimator,
    features: pd.DataFrame,
    labels: pd.Series,
    threshold: float = 0.5,
) -> Dict[str, float]:
    """Evaluate binary classifier with common classification metrics."""
    prob = predict_positive_probability(model, features)
    pred = (prob >= threshold).astype(int)

    precision_curve, recall_curve, _ = precision_recall_curve(labels, prob)

    return {
        "Accuracy": accuracy_score(labels, pred),
        "Precision": precision_score(labels, pred, zero_division=0),
        "Recall": recall_score(labels, pred, zero_division=0),
        "F1": f1_score(labels, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(labels, prob),
        "PR_AUC": auc(recall_curve, precision_curve),
    }


def fit_and_evaluate_models(
    models: Dict[str, BaseEstimator],
) -> Tuple[Dict[str, BaseEstimator], pd.DataFrame]:
    """Fit candidate models and return evaluation table."""
    fitted_models = {}
    records = []

    for name, model in models.items():
        print(f"Training model: {name}")
        model.fit(X_train, y_train)
        fitted_models[name] = model

        metrics = evaluate_binary_classifier(model, X_test, y_test)
        metrics["Model"] = name
        records.append(metrics)

    result_df = pd.DataFrame(records).set_index("Model")
    result_df = result_df.sort_values(by="ROC_AUC", ascending=False)
    result_df.to_csv(TABLE_DIR / "model_comparison.csv", encoding="utf-8-sig")
    return fitted_models, result_df


models = build_models()
fitted_models, model_results = fit_and_evaluate_models(models)
model_results

## 8. 交叉验证

使用三折 StratifiedKFold 评估模型排序能力的稳定性。

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

cv_records = []
for name in ["LogisticRegression_SMOTE", "GradientBoosting"]:
    print(f"Cross-validating: {name}")
    scores = cross_val_score(
        fitted_models[name],
        X,
        y,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
    )
    cv_records.append({
        "Model": name,
        "CV_ROC_AUC_Mean": scores.mean(),
        "CV_ROC_AUC_Std": scores.std(),
    })

cv_results = pd.DataFrame(cv_records).set_index("Model")
cv_results.to_csv(TABLE_DIR / "cross_validation_results.csv", encoding="utf-8-sig")
cv_results

## 9. ROC 曲线与 PR 曲线

In [ ]:
plt.figure(figsize=(7, 5))
for name, model in fitted_models.items():
    prob = predict_positive_probability(model, X_test)
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_value = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} AUC={roc_value:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.title("不同模型 ROC 曲线")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
save_current_figure("05_roc_curves.png")
plt.show()

plt.figure(figsize=(7, 5))
for name, model in fitted_models.items():
    prob = predict_positive_probability(model, X_test)
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, prob)
    pr_value = auc(recall_curve, precision_curve)
    plt.plot(recall_curve, precision_curve, label=f"{name} PR-AUC={pr_value:.3f}")

baseline = y_test.mean()
plt.axhline(baseline, linestyle="--", label=f"Positive baseline={baseline:.3f}")
plt.title("不同模型 PR 曲线")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
save_current_figure("06_pr_curves.png")
plt.show()

## 10. 最优模型选择与阈值优化

在客户流失预警中，业务目标通常更重视“找出更多潜在流失客户”，因此在默认阈值 0.5 之外，额外基于 PR 曲线寻找 F1 最优阈值。

In [ ]:
BEST_MODEL_NAME = model_results.index[0]
best_model = fitted_models[BEST_MODEL_NAME]

best_prob = predict_positive_probability(best_model, X_test)
precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, best_prob)

f1_scores = 2 * precision_curve[:-1] * recall_curve[:-1] / (
    precision_curve[:-1] + recall_curve[:-1] + 1e-12
)
best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = f1_scores.max()

default_metrics = evaluate_binary_classifier(best_model, X_test, y_test, threshold=0.5)
optimized_metrics = evaluate_binary_classifier(best_model, X_test, y_test, threshold=best_threshold)

threshold_results = pd.DataFrame([
    {"Threshold": 0.5, **default_metrics},
    {"Threshold": best_threshold, **optimized_metrics},
])
threshold_results.to_csv(TABLE_DIR / "threshold_optimization.csv", index=False, encoding="utf-8-sig")

print(f"Best model: {BEST_MODEL_NAME}")
print(f"Best F1 threshold: {best_threshold:.3f}, Best F1: {best_f1:.4f}")
threshold_results

In [ ]:
optimized_pred = (best_prob >= best_threshold).astype(int)
cm = confusion_matrix(y_test, optimized_pred)

fig, ax = plt.subplots(figsize=(5, 4))
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Churn", "Churn"],
)
display.plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
ax.set_title(f"{BEST_MODEL_NAME} 混淆矩阵（阈值={best_threshold:.3f}）")
save_current_figure("07_confusion_matrix_optimized.png")
plt.show()

## 11. Gradient Boosting 超参数敏感性分析

分析 `n_estimators`、`learning_rate` 和 `max_depth` 对结果的影响。  
此部分用于对应报告中的超参数敏感性分析。

In [ ]:
sensitivity_configs = [
    {"n_estimators": 80, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 120, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 160, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 120, "learning_rate": 0.03, "max_depth": 3},
    {"n_estimators": 120, "learning_rate": 0.08, "max_depth": 3},
    {"n_estimators": 120, "learning_rate": 0.10, "max_depth": 3},
    {"n_estimators": 120, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 120, "learning_rate": 0.05, "max_depth": 4},
]

sensitivity_records = []

for params in sensitivity_configs:
    model = Pipeline(steps=[
        ("preprocess", build_preprocessor()),
        ("model", GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **params,
        )),
    ])
    model.fit(X_train, y_train)
    metrics = evaluate_binary_classifier(model, X_test, y_test)
    sensitivity_records.append({**params, **metrics})

sensitivity_df = pd.DataFrame(sensitivity_records)
sensitivity_df.to_csv(TABLE_DIR / "gb_hyperparameter_sensitivity.csv", index=False, encoding="utf-8-sig")
sensitivity_df

In [ ]:
plt.figure(figsize=(7, 4))
plot_df = sensitivity_df.copy()
plot_df["config"] = [
    f"n={r.n_estimators}, lr={r.learning_rate}, d={r.max_depth}"
    for r in plot_df.itertuples()
]
sns.barplot(data=plot_df, x="config", y="ROC_AUC")
plt.xticks(rotation=45, ha="right")
plt.title("Gradient Boosting 超参数敏感性：ROC-AUC")
plt.xlabel("参数组合")
plt.ylabel("ROC-AUC")
save_current_figure("08_hyperparameter_sensitivity_roc_auc.png")
plt.show()

## 12. 特征重要性分析

In [ ]:
def get_feature_names(preprocessor: ColumnTransformer) -> List[str]:
    """Get feature names after preprocessing."""
    names: List[str] = []

    if numeric_features:
        names.extend(numeric_features)

    if categorical_features:
        onehot = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        cat_names = onehot.get_feature_names_out(categorical_features).tolist()
        names.extend(cat_names)

    return names


best_preprocessor = best_model.named_steps["preprocess"]
best_estimator = best_model.named_steps["model"]
feature_names = get_feature_names(best_preprocessor)

if hasattr(best_estimator, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": best_estimator.feature_importances_,
    }).sort_values("Importance", ascending=False)
else:
    raise AttributeError("当前最优模型不支持 feature_importances_。")

importance_df.to_csv(TABLE_DIR / "feature_importance.csv", index=False, encoding="utf-8-sig")
importance_df.head(20)

In [ ]:
plt.figure(figsize=(8, 7))
top_importance = importance_df.head(20)
sns.barplot(data=top_importance, y="Feature", x="Importance")
plt.title("Top-20 特征重要性")
plt.xlabel("Importance")
plt.ylabel("Feature")
save_current_figure("09_feature_importance_top20.png")
plt.show()

## 13. SHAP 可解释性分析

若环境中未安装 SHAP，请先运行：

```bash
pip install shap
```

SHAP 计算可能需要一定时间。为了保证课堂复现效率，默认使用测试集前 1000 条样本计算全局解释。

In [ ]:
try:
    import shap

    shap_available = True
    print("SHAP imported successfully.")
except ImportError:
    shap_available = False
    print("SHAP is not installed. Please run: pip install shap")

In [ ]:
if shap_available:
    X_test_processed = best_preprocessor.transform(X_test)
    shap_sample_size = min(1000, X_test_processed.shape[0])
    X_shap = X_test_processed[:shap_sample_size]

    explainer = shap.TreeExplainer(best_estimator)
    shap_values = explainer.shap_values(X_shap)

    if isinstance(shap_values, list):
        shap_values_for_positive = shap_values[1]
    else:
        shap_values_for_positive = shap_values

    mean_abs_shap = np.abs(shap_values_for_positive).mean(axis=0)
    shap_importance_df = pd.DataFrame({
        "Feature": feature_names,
        "MeanAbsSHAP": mean_abs_shap,
    }).sort_values("MeanAbsSHAP", ascending=False)

    shap_importance_df.to_csv(TABLE_DIR / "shap_global_importance.csv", index=False, encoding="utf-8-sig")
    display(shap_importance_df.head(20))

    plt.figure(figsize=(8, 7))
    sns.barplot(data=shap_importance_df.head(20), y="Feature", x="MeanAbsSHAP")
    plt.title("Top-20 SHAP 全局重要性")
    plt.xlabel("Mean |SHAP value|")
    plt.ylabel("Feature")
    save_current_figure("10_shap_global_importance_top20.png")
    plt.show()
else:
    print("Skip SHAP analysis because shap is not installed.")

## 14. K-Means 客户价值细分

聚类特征选择：

- `tenure`
- `MonthlyCharges`
- `TotalCharges`
- `ChargePerTenure`

聚类后将客户分群结果与流失标签关联，分析不同群体的风险与价值。

In [ ]:
cluster_features = ["tenure", "MonthlyCharges", "TotalCharges", "ChargePerTenure"]

cluster_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)),
])

df_clustered = df.copy()
df_clustered["CustomerSegment"] = cluster_pipeline.fit_predict(df_clustered[cluster_features])

segment_summary = (
    df_clustered.groupby("CustomerSegment")
    .agg(
        Customers=("customerID", "count"),
        ChurnRate=("ChurnLabel", "mean"),
        AvgTenure=("tenure", "mean"),
        AvgMonthlyCharges=("MonthlyCharges", "mean"),
        AvgTotalCharges=("TotalCharges", "mean"),
    )
    .sort_values("ChurnRate", ascending=False)
)

segment_summary["ChurnRate"] = segment_summary["ChurnRate"].map(lambda x: f"{x:.2%}")
segment_summary.to_csv(TABLE_DIR / "customer_segment_summary.csv", encoding="utf-8-sig")
segment_summary

In [ ]:
plt.figure(figsize=(8, 5))
plot_cluster = df_clustered.copy()
sns.scatterplot(
    data=plot_cluster,
    x="tenure",
    y="MonthlyCharges",
    hue="CustomerSegment",
    style="Churn",
    alpha=0.75,
    palette="tab10",
)
plt.title("客户价值细分可视化")
plt.xlabel("在网时长（月）")
plt.ylabel("月费用")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
save_current_figure("11_customer_segmentation.png")
plt.show()

## 15. 单客户流失风险推理函数

该函数用于输入单个客户信息，输出流失概率和风险等级。  
实际部署时，可将该函数封装为简单 API 或 Streamlit 页面。

In [ ]:
def predict_single_customer_churn(
    customer_record: Dict[str, object],
    model: BaseEstimator = best_model,
    threshold: float = float(best_threshold),
) -> Dict[str, object]:
    """Predict churn risk for one customer record."""
    customer_df = pd.DataFrame([customer_record])

    if "TotalCharges" in customer_df.columns:
        customer_df["TotalCharges"] = pd.to_numeric(
            customer_df["TotalCharges"], errors="coerce"
        ).fillna(0.0)

    if "tenure" in customer_df.columns and "TotalCharges" in customer_df.columns:
        customer_df["AvgMonthlyCharge"] = np.where(
            customer_df["tenure"] > 0,
            customer_df["TotalCharges"] / customer_df["tenure"],
            0.0,
        )

    if "MonthlyCharges" in customer_df.columns and "tenure" in customer_df.columns:
        customer_df["ChargePerTenure"] = (
            customer_df["MonthlyCharges"] / (customer_df["tenure"] + 1)
        )

    if "tenure" in customer_df.columns:
        customer_df["TenureSegment"] = pd.cut(
            customer_df["tenure"],
            bins=[-1, 6, 12, 24, 48, 72],
            labels=["0-6", "7-12", "13-24", "25-48", "49-72"],
        ).astype(str)

    for column in X.columns:
        if column not in customer_df.columns:
            customer_df[column] = np.nan
    customer_df = customer_df[X.columns]

    probability = float(predict_positive_probability(model, customer_df)[0])
    prediction = int(probability >= threshold)

    if probability >= 0.70:
        risk_level = "高风险"
    elif probability >= 0.40:
        risk_level = "中风险"
    else:
        risk_level = "低风险"

    return {
        "ChurnProbability": probability,
        "Prediction": "Churn" if prediction == 1 else "Not Churn",
        "RiskLevel": risk_level,
        "Threshold": threshold,
    }


example_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 2,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 80.0,
    "TotalCharges": 160.0,
}

predict_single_customer_churn(example_customer)

## 16. 保存模型与实验结果

运行完成后，所有输出将保存在：

```text
outputs/
├── figures/
├── models/
└── tables/
```

In [ ]:
joblib.dump(best_model, MODEL_DIR / "best_churn_model.joblib")
joblib.dump(cluster_pipeline, MODEL_DIR / "customer_segmentation_kmeans.joblib")

summary = {
    "best_model": BEST_MODEL_NAME,
    "best_threshold": float(best_threshold),
    "default_metrics": default_metrics,
    "optimized_metrics": optimized_metrics,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
}

pd.Series(summary).to_json(OUTPUT_DIR / "experiment_summary.json", force_ascii=False, indent=2)

print(f"Best model saved to: {MODEL_DIR / 'best_churn_model.joblib'}")
print(f"KMeans model saved to: {MODEL_DIR / 'customer_segmentation_kmeans.joblib'}")
print(f"Experiment summary saved to: {OUTPUT_DIR / 'experiment_summary.json'}")

## 17. 实验结论

本 Notebook 实现了电信客户流失预测与客户价值细分的完整流程。  
从实验设计看，本项目不仅完成了分类建模，还进一步补充了阈值优化、超参数敏感性分析、SHAP 可解释性分析和 K-Means 客户分群，因此能够支撑报告中的主要结论。

最终提交时建议同时提交：

1. 本 Notebook 文件；
2. 报告 PDF；
3. 答辩 PPT；
4. requirements.txt；
5. README.md；
6. 数据说明或数据下载方式。